# Intro to Model and Data Drift

Once a model is deployed, the data it sees can slowly change. Sometimes the feature distribution shifts, sometimes the target behavior changes, and sometimes both happen together. This notebook shows how to compare two taxi datasets and turn that intuition into measurable drift signals.


```mermaid
flowchart LR
    A["Training month"] --> C["Compare distributions"]
    B["Current month"] --> C
    C --> D["Feature drift"]
    C --> E["Target drift"]
    C --> F["Prediction drift"]
    D --> G["Investigate or monitor"]
    E --> G
    F --> G
```


**Example:**

You've built a model that tries to predict how much a person will spend in an online webshop, using the age and the gender of a person. The training dataset mostly consists of females, with the average age of all customers being 35. The model learns that mostly older people and women buy something more often. After monitoring the model's predictive performance, you notice that it's starting to decrease... But why? 

After doing some research you discovered that the webshop now has younger customers: the average age has dropped from 35 years to 25 years. Furthermore, the live data shows that currently not the women are majority, but the men!

So, the original relationship between the features that the model has learned from the original training dataset isn't valid anymore on the real-time incoming data, which causes the model to lose its predictive power. **This loss in predictive power is called model drift and the change in the data is called data drift**.

Let's have a look at the NY Taxi dataset.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from dotenv import load_dotenv

Let's start by loading the datasets. The first one is the dataset we train the model on:

In [ ]:
year = 2024
month = 1
color = "green"

df_24_1 = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

The second one is the dataset we compare the first one to:

In [ ]:
year = 2025
month = 2
color = "green"

data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
data_path.parent.mkdir(exist_ok=True)

if not data_path.exists():
    urlretrieve(
        f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
        data_path,
    )

df_25_2 = pd.read_parquet(data_path)

In [ ]:
features = [
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]
target = "duration"

In [ ]:
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] < 8)]
    df = df[features + [target]]
    return df

In [ ]:
df_24_1 = calculate_trip_duration_in_minutes(df_24_1)
df_25_2 = calculate_trip_duration_in_minutes(df_25_2)

if len(df_24_1) < len(df_25_2):
    print("Trimming df_25_2 to match length of df_24_1")
    df_25_2 = df_25_2.sample(frac=1, random_state=42).reset_index(drop=True)
    df_25_2 = df_25_2.iloc[: len(df_24_1)]
elif len(df_24_1) > len(df_25_2):
    print("Trimming df_24_1 to match length of df_25_2")
    df_24_1 = df_24_1.sample(frac=1, random_state=42).reset_index(drop=True)
    df_24_1 = df_24_1.iloc[: len(df_25_2)]
else:
    print("Both DataFrames have the same length")

### Now let's compare the data

In [ ]:
df_24_1.describe()

In [ ]:
df_25_2.describe()

In [ ]:
fig, ax = plt.subplots(1, len(df_24_1.columns), figsize=(3 * len(df_24_1.columns), 3))
for i, var in enumerate(df_24_1):
    df_24_1[var].hist(ax=ax[i])
    ax[i].set_title(var)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, len(df_25_2.columns), figsize=(3 * len(df_25_2.columns), 3))
for i, var in enumerate(df_25_2):
    df_25_2[var].hist(ax=ax[i])
    ax[i].set_title(var)
plt.show()

We see small changes in nearly all features. But we can also calculate the difference between the two datasets with:
- **Wasserstein distance** for numerical features;
- **Jensen-Shannon distance** for categorical features.

For categorical features, we first convert location IDs into normalized frequency distributions, then compare those distributions. The lower the value, the more similar the two datasets are.

Let's calculate the distance between the features of the two datasets:

In [ ]:
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance


def drift_wasserstein_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between two numerical distributions."""
    return float(wasserstein_distance(var_train, var_test))


def drift_jensenshannon_test(var_train: pd.Series, var_test: pd.Series) -> float:
    """Measure distance between categorical probability distributions."""
    categories = sorted(set(var_train.dropna()) | set(var_test.dropna()))
    train_distribution = var_train.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    test_distribution = var_test.value_counts(normalize=True).reindex(
        categories, fill_value=0
    )
    return float(jensenshannon(train_distribution, test_distribution))

In [ ]:
df_drift = pd.DataFrame(
    {
        "feature": pd.Series(dtype="object"),
        "distance": pd.Series(dtype="float64"),
        "type": pd.Series(dtype="object"),
    }
)

cat_features = ["PULocationID", "DOLocationID"]
num_features = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]

for feature in num_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_wasserstein_test(
                            df_24_1[feature], df_25_2[feature]
                        ),
                        "type": "Wasserstein",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

for feature in cat_features:
    df_drift = pd.concat(
        [
            df_drift,
            pd.DataFrame(
                [
                    {
                        "feature": feature,
                        "distance": drift_jensenshannon_test(
                            df_24_1[feature], df_25_2[feature]
                        ),
                        "type": "Jensen-Shannon",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

There are more **tests to detect drift**, but this is a good start. If you want to learn more about **drift detection**, check out this [article](https://www.evidentlyai.com/blog/data-drift-detection-large-datasets). In practice, thresholds should be calibrated for the data and business context. For this lesson, we use `0.1` as a simple rule of thumb so we can practice turning distances into a drift flag:

In [ ]:
df_drift["True/False Drift"] = df_drift["distance"] > 0.1
df_drift

It seems we can confirm what we saw in the histograms: there is drift in half of our features. Using this method **we can also detect drift in the target variable**. Let's have a look at the target variable:

In [ ]:
drift_wasserstein_test(df_24_1[target], df_25_2[target])

So we also have a drift in our target variable.

We can also check whether the model predictions themselves are drifting. For that step we load the registered model created in notebook 1 and score both datasets with the same feature set.


In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

In [ ]:
def load_model(model_name):
    alias = "production"
    model_uri = f"models:/{model_name}@{alias}"
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
model_name = "green-taxi-ride-duration"

loaded_model = load_model(model_name)

In [ ]:
df_24_1["prediction"] = loaded_model.predict(df_24_1[features])

In [ ]:
df_25_2["prediction"] = loaded_model.predict(df_25_2[features])

Now we can calculate the drift in the predictions:

In [ ]:
drift_wasserstein_test(df_24_1["prediction"], df_25_2["prediction"])

And also here we can see drift. That does not automatically mean the model is unusable, but it is a strong signal that the training distribution and the current production-like batch are no longer aligned. In practice, this is the kind of evidence that can trigger retraining or a closer investigation.


In the next notebook we will use Evidently to automate these checks and package them into a reusable report.
